- 3 subt params
- 2 pf params

In [16]:
import numpy as np
import scipy as sp
from matplotlib import pyplot as plt

import pandas as pd
import seaborn as sns
import os
from statsmodels.stats.multitest import fdrcorrection
# from google.colab import drive
# drive.mount('/content/drive')

from scipy.stats import zscore

import statsmodels.api as sm

# !pip install scikit-bio

In [17]:
# mixed_batch_data = pd.read_csv('/content/drive/MyDrive/curvefit_traits_merged_linearpf_onlyneccols.csv', index_col='subID')
mixed_batch_data = pd.read_csv('../../../data/subtlety_playfight_data/curvefit_traits_merged_linearpf_onlyneccols_fight.csv', index_col='subID')
mixed_batch_data.drop('Unnamed: 0', axis=1, inplace=True)
mixed_batch_data

,PSE_subt,range_subt,bias_subt,sigma_subt,PSE_pf,slope_pf,range_pf,bias_pf,social_skill,attn_switch,...,comm,posAffect,negAffect,neuroticism,extraversion,openness,agreeableness,conscientiousness,loneliness,nfriends
subID,,,,,,,,,,,,,,,,,,,,,
30002.0,0.552951,0.900821,0.315504,0.103360,0.513054,0.784286,0.784286,0.452539,21.0,23.0,...,16.000000,20.0,0.0,38.0,28.000000,31.000000,34.0,39.0,56.0,3.0
30004.0,0.276896,0.805834,0.999458,0.076866,0.872907,0.469286,0.469286,0.170256,21.0,20.0,...,15.000000,25.0,11.0,38.0,26.181818,50.000000,50.0,46.0,56.0,3.0
30005.0,0.320583,0.710131,0.861730,0.115353,0.501696,0.842143,0.842143,0.490950,17.0,29.0,...,16.000000,20.0,21.0,51.0,39.000000,40.000000,43.0,19.0,41.0,5.0
30006.0,0.464823,0.989396,0.889572,0.079221,0.741046,0.674587,0.674587,0.000307,8.0,10.0,...,3.000000,25.0,0.0,18.0,41.000000,31.000000,44.0,57.0,30.0,4.0
30008.0,0.325118,0.582252,0.692710,0.172585,0.677591,0.511071,0.511071,0.314366,5.0,15.0,...,12.222222,28.0,4.0,27.0,46.000000,42.545455,46.0,50.0,45.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30344.0,0.361233,0.655715,0.794480,0.135840,0.492859,0.566786,0.566786,0.509343,27.0,18.0,...,16.000000,14.0,0.0,25.0,29.000000,49.000000,35.0,56.0,41.0,6.0
30345.0,0.475067,0.833333,1.000000,0.013915,0.639965,0.781136,0.781136,0.000457,15.0,14.0,...,5.000000,15.0,15.0,36.0,23.000000,37.000000,45.0,54.0,72.0,1.0
30346.0,0.426850,0.999914,0.961644,0.045409,0.412504,0.786429,0.786429,0.822185,16.0,25.0,...,8.000000,27.0,6.0,49.0,36.000000,36.000000,43.0,44.0,56.0,3.0


In [18]:
mixed_batch_data.isna().sum()

PSE_subt             0
range_subt           0
bias_subt            0
sigma_subt           0
PSE_pf               0
slope_pf             0
range_pf             0
bias_pf              0
social_skill         0
attn_switch          0
img                  0
attn_to_det          0
comm                 0
posAffect            0
negAffect            0
neuroticism          0
extraversion         0
openness             0
agreeableness        0
conscientiousness    0
loneliness           0
nfriends             1
dtype: int64

Looks like one subject is missing a value for `nfriends`. Let's just drop them as this will cause headaches later

In [19]:
mixed_batch_data = mixed_batch_data.dropna()
mixed_batch_data.shape

(274, 22)

In [20]:
mixed_batch_data.columns

Index(['PSE_subt', 'range_subt', 'bias_subt', 'sigma_subt', 'PSE_pf',
       'slope_pf', 'range_pf', 'bias_pf', 'social_skill', 'attn_switch', 'img',
       'attn_to_det', 'comm', 'posAffect', 'negAffect', 'neuroticism',
       'extraversion', 'openness', 'agreeableness', 'conscientiousness',
       'loneliness', 'nfriends'],
      dtype='object')

In [21]:
curve_param_cols = [
                  'PSE_subt',
                  'range_subt',
                  'bias_subt',
                  'range_pf',
                  'bias_pf'
                    ]
subt_param_cols = [
                  'PSE_subt',
                  'range_subt',
                  'bias_subt',
                   ]
pf_param_cols = [
                 'range_pf',
                 'bias_pf'
                 ]
trait_cols = [
              'social_skill', 'attn_switch', 'img', 'attn_to_det', 'comm',
              'posAffect',
              'negAffect',
              'neuroticism', 'extraversion', 'openness', 'agreeableness', 'conscientiousness',
              'loneliness',
              'nfriends'
       ]

social_cols = [
              'social_skill', 'attn_switch', 'img', 'attn_to_det', 'comm',
              'loneliness',
              'nfriends'
              ]

In [22]:
results_loc= '../../../results/subtlety_playfight_mixed/trait-beh/extra_analyses/'

In [23]:
# first, super important to z-score all the features in order to minimize the impact of different features being measured on different scales (just like PCA)
mixed_batch_data_zscored = mixed_batch_data.apply(zscore)
mixed_batch_data_zscored

,PSE_subt,range_subt,bias_subt,sigma_subt,PSE_pf,slope_pf,range_pf,bias_pf,social_skill,attn_switch,...,comm,posAffect,negAffect,neuroticism,extraversion,openness,agreeableness,conscientiousness,loneliness,nfriends
subID,,,,,,,,,,,,,,,,,,,,,
30002.0,1.587100,1.139946,-2.182466,0.144545,-0.206179,0.740744,0.802900,0.052813,1.249255,1.403779,...,1.044306,-0.214096,-1.031925,0.328778,-1.032702,-1.533105,-1.890716,-0.771265,0.884033,-0.304004
30004.0,-0.836964,0.619923,1.390323,-0.452518,2.089832,-0.732085,-0.731401,-0.970100,1.249255,0.813345,...,0.862296,0.379919,0.470275,0.328778,-1.249254,1.402822,1.569992,0.078101,0.884033,-0.304004
30005.0,-0.453342,0.095984,0.670868,0.414827,-0.278646,1.011263,1.084710,0.192006,0.604630,2.584649,...,1.044306,-0.214096,1.835912,1.543459,0.277433,-0.142403,0.055932,-3.198024,-0.300924,0.449816
30006.0,0.813243,1.624863,0.816307,-0.399448,1.248505,0.227830,0.268579,-1.585944,-0.845776,-1.154772,...,-1.321823,0.379919,-1.031925,-1.539962,0.515640,-1.533105,0.272226,1.412818,-1.169892,0.072906
30008.0,-0.413524,-0.604109,-0.212045,1.704615,0.843637,-0.536710,-0.527871,-0.447886,-1.329245,-0.170714,...,0.356713,0.736328,-0.485670,-0.699029,1.111156,0.250927,0.704815,0.563453,0.015065,0.449816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30344.0,-0.096389,-0.201924,0.319575,0.876526,-0.335034,-0.276210,-0.256498,0.258657,2.216193,0.419721,...,1.044306,-0.926914,-1.031925,-0.885903,-0.913599,1.248300,-1.674422,1.291480,-0.300924,0.826726
30345.0,0.903194,0.770473,1.393153,-1.871185,0.603566,0.726018,0.787560,-1.585402,0.282318,-0.367525,...,-0.957803,-0.808111,1.016530,0.141904,-1.628218,-0.605970,0.488521,1.048804,2.147987,-1.057824
30346.0,0.479799,1.682444,1.192794,-1.161440,-0.847731,0.750763,0.813338,1.392306,0.443474,1.797403,...,-0.411773,0.617525,-0.212543,1.356585,-0.079876,-0.760493,0.055932,-0.164575,0.884033,-0.304004


# Linear regression 

In [24]:
len(trait_cols)

14

In [1]:
from pymer4.models import Lmer, Lm

ModuleNotFoundError: No module named 'pymer4'

In [26]:
mixed_batch_data_zscored.head()

,PSE_subt,range_subt,bias_subt,sigma_subt,PSE_pf,slope_pf,range_pf,bias_pf,social_skill,attn_switch,...,comm,posAffect,negAffect,neuroticism,extraversion,openness,agreeableness,conscientiousness,loneliness,nfriends
subID,,,,,,,,,,,,,,,,,,,,,
30002.0,1.587100,1.139946,-2.182466,0.144545,-0.206179,0.740744,0.802900,0.052813,1.249255,1.403779,...,1.044306,-0.214096,-1.031925,0.328778,-1.032702,-1.533105,-1.890716,-0.771265,0.884033,-0.304004
30004.0,-0.836964,0.619923,1.390323,-0.452518,2.089832,-0.732085,-0.731401,-0.970100,1.249255,0.813345,...,0.862296,0.379919,0.470275,0.328778,-1.249254,1.402822,1.569992,0.078101,0.884033,-0.304004
30005.0,-0.453342,0.095984,0.670868,0.414827,-0.278646,1.011263,1.084710,0.192006,0.604630,2.584649,...,1.044306,-0.214096,1.835912,1.543459,0.277433,-0.142403,0.055932,-3.198024,-0.300924,0.449816
30006.0,0.813243,1.624863,0.816307,-0.399448,1.248505,0.227830,0.268579,-1.585944,-0.845776,-1.154772,...,-1.321823,0.379919,-1.031925,-1.539962,0.515640,-1.533105,0.272226,1.412818,-1.169892,0.072906
30008.0,-0.413524,-0.604109,-0.212045,1.704615,0.843637,-0.536710,-0.527871,-0.447886,-1.329245,-0.170714,...,0.356713,0.736328,-0.485670,-0.699029,1.111156,0.250927,0.704815,0.563453,0.015065,0.449816


In [27]:
from scipy import stats

In [28]:
x = mixed_batch_data_zscored['openness']
y = mixed_batch_data_zscored['range_pf']
stats.pearsonr(x,y)

PearsonRResult(statistic=np.float64(-0.14517099076336462), pvalue=np.float64(0.01618245383746685))

In [30]:
x = mixed_batch_data_zscored['comm']
y = mixed_batch_data_zscored['bias_subt']
stats.pearsonr(x,y)

PearsonRResult(statistic=np.float64(-0.12728067802299373), pvalue=np.float64(0.03522083935901831))

# subtlety + playfight terms

In [14]:
param_list = ['PSE_subt','range_subt','bias_subt','range_pf','bias_pf']

all_ps = np.full((14,len(param_list)),np.nan) # for MCC
for t,trait in enumerate(trait_cols):
    print(f'\nTRAIT {trait}')
    model = Lm(f"{trait} ~ PSE_subt + range_subt + bias_subt + range_pf + bias_pf", data=mixed_batch_data_zscored)
    print(model.fit())

    for p,param in enumerate(param_list): # for MCC
        all_ps[t,p] = model.coefs['P-val'][param]


TRAIT social_skill
Formula: social_skill~PSE_subt+range_subt+bias_subt+range_pf+bias_pf

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of observations: 274	 R^2: 0.020	 R^2_adj: 0.002

Log-likelihood: -385.953 	 AIC: 783.906	 BIC: 805.585

Fixed effects:

            Estimate  2.5_ci  97.5_ci     SE   DF  T-stat  P-val Sig
Intercept     -0.000  -0.119    0.119  0.060  268  -0.000  1.000    
PSE_subt      -0.044  -0.211    0.123  0.085  268  -0.521  0.603    
range_subt     0.139  -0.023    0.300  0.082  268   1.692  0.092   .
bias_subt     -0.137  -0.298    0.024  0.082  268  -1.677  0.095   .
range_pf      -0.005  -0.144    0.135  0.071  268  -0.066  0.947    
bias_pf        0.014  -0.106    0.134  0.061  268   0.226  0.822    

TRAIT attn_switch
Formula: attn_switch~PSE_subt+range_subt+bias_subt+range_pf+bias_pf

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of obs

In [15]:
## MCC
result_bool = np.full_like(all_ps,np.nan)
corrected_p = np.full_like(all_ps,np.nan)
for p, param in enumerate(param_list):
    result_bool[:,p], corrected_p[:,p] = fdrcorrection(all_ps[:,p])
    print(param, np.where(result_bool[:,p])[0], np.array(trait_cols)[np.where(result_bool[:,p])[0]])
# i.e., openness-range_pf is the only relationship that survives MCC

PSE_subt [] []
range_subt [] []
bias_subt [] []
range_pf [9] ['openness']
bias_pf [] []


# only subtlety terms

In [70]:
param_list = ['PSE_subt','range_subt','bias_subt']

all_ps_subt = np.full((14,len(param_list)),np.nan) # for MCC
for t,trait in enumerate(trait_cols):
    print(f'\nTRAIT {trait}')
    model = Lm(f"{trait} ~ PSE_subt + range_subt + bias_subt", data=mixed_batch_data_zscored)
    print(model.fit())

    for p,param in enumerate(param_list): # for MCC
        all_ps_subt[t,p] = model.coefs['P-val'][param]


TRAIT social_skill
Formula: social_skill~PSE_subt+range_subt+bias_subt

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of observations: 274	 R^2: 0.020	 R^2_adj: 0.009

Log-likelihood: -385.981 	 AIC: 779.962	 BIC: 794.414

Fixed effects:

            Estimate  2.5_ci  97.5_ci     SE   DF  T-stat  P-val Sig
Intercept     -0.000  -0.119    0.119  0.060  270  -0.000  1.000    
PSE_subt      -0.045  -0.211    0.121  0.084  270  -0.533  0.595    
range_subt     0.137  -0.017    0.290  0.078  270   1.755  0.080   .
bias_subt     -0.137  -0.291    0.017  0.078  270  -1.757  0.080   .

TRAIT attn_switch
Formula: attn_switch~PSE_subt+range_subt+bias_subt

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of observations: 274	 R^2: 0.003	 R^2_adj: -0.008

Log-likelihood: -388.426 	 AIC: 784.853	 BIC: 799.305

Fixed effects:

            Estimate  2.5_ci  97.5_ci     SE   DF  T-sta

In [71]:
## MCC
result_bool = np.full_like(all_ps,np.nan)
corrected_p = np.full_like(all_ps,np.nan)
for p, param in enumerate(param_list):
    result_bool[:,p], corrected_p[:,p] = fdrcorrection(all_ps[:,p])
    print(param, np.where(result_bool[:,p])[0], np.array(trait_cols)[np.where(result_bool[:,p])[0]])
# i.e., openness-range_pf is the only relationship that survives MCC

PSE_subt [] []
range_subt [] []
bias_subt [] []


# only playfight terms

In [72]:
param_list = ['range_pf','bias_pf']

all_ps = np.full((14,len(param_list)),np.nan) # for MCC
for t,trait in enumerate(trait_cols):
    print(f'\nTRAIT {trait}')
    model = Lm(f"{trait} ~ range_pf + bias_pf", data=mixed_batch_data_zscored)
    print(model.fit())

    for p,param in enumerate(param_list): # for MCC
        all_ps[t,p] = model.coefs['P-val'][param]


TRAIT social_skill
Formula: social_skill~range_pf+bias_pf

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of observations: 274	 R^2: 0.000	 R^2_adj: -0.007

Log-likelihood: -388.788 	 AIC: 783.576	 BIC: 794.415

Fixed effects:

           Estimate  2.5_ci  97.5_ci     SE   DF  T-stat  P-val Sig
Intercept    -0.000  -0.120    0.120  0.061  271  -0.000  1.000    
range_pf      0.003  -0.117    0.123  0.061  271   0.047  0.963    
bias_pf       0.000  -0.120    0.120  0.061  271   0.001  0.999    

TRAIT attn_switch
Formula: attn_switch~range_pf+bias_pf

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of observations: 274	 R^2: 0.008	 R^2_adj: 0.001

Log-likelihood: -387.690 	 AIC: 781.381	 BIC: 792.220

Fixed effects:

           Estimate  2.5_ci  97.5_ci     SE   DF  T-stat  P-val Sig
Intercept     0.000  -0.119    0.119  0.061  271   0.000  1.000    
range_pf      0.082

In [76]:
all_ps.shape

(14, 2)

In [73]:
## MCC
result_bool = np.full_like(all_ps,np.nan)
corrected_p = np.full_like(all_ps,np.nan)
for p, param in enumerate(param_list):
    result_bool[:,p], corrected_p[:,p] = fdrcorrection(all_ps[:,p])
    print(param, np.where(result_bool[:,p])[0], np.array(trait_cols)[np.where(result_bool[:,p])[0]])
# i.e., openness-range_pf is the only relationship that survives MCC

range_pf [] []
bias_pf [] []


 # openness vs. params
 - does combining both expts improve the model?

In [62]:
trait = 'openness'

In [63]:
model = Lm(f"{trait} ~ PSE_subt + range_subt + bias_subt", data=mixed_batch_data_zscored)
print(model.fit())

Formula: openness~PSE_subt+range_subt+bias_subt

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of observations: 274	 R^2: 0.002	 R^2_adj: -0.009

Log-likelihood: -388.489 	 AIC: 784.977	 BIC: 799.430

Fixed effects:

            Estimate  2.5_ci  97.5_ci     SE   DF  T-stat  P-val Sig
Intercept     -0.000  -0.120    0.120  0.061  270  -0.000  1.000    
PSE_subt       0.042  -0.126    0.210  0.085  270   0.493  0.622    
range_subt    -0.030  -0.185    0.125  0.078  270  -0.383  0.702    
bias_subt      0.061  -0.095    0.216  0.079  270   0.769  0.443    


In [64]:
model = Lm(f"{trait} ~  range_pf + bias_pf", data=mixed_batch_data_zscored)
print(model.fit())

Formula: openness~range_pf+bias_pf

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of observations: 274	 R^2: 0.023	 R^2_adj: 0.015

Log-likelihood: -385.648 	 AIC: 777.295	 BIC: 788.135

Fixed effects:

           Estimate  2.5_ci  97.5_ci    SE   DF  T-stat  P-val Sig
Intercept    -0.000  -0.118    0.118  0.06  271  -0.000  1.000    
range_pf     -0.143  -0.262   -0.025  0.06  271  -2.386  0.018   *
bias_pf      -0.040  -0.158    0.078  0.06  271  -0.665  0.507    


In [65]:
model = Lm(f"{trait} ~ PSE_subt + range_subt + bias_subt + range_pf + bias_pf", data=mixed_batch_data_zscored)
print(model.fit())

Formula: openness~PSE_subt+range_subt+bias_subt+range_pf+bias_pf

Family: gaussian	 Estimator: OLS

Std-errors: non-robust	CIs: standard 95%	Inference: parametric 

Number of observations: 274	 R^2: 0.038	 R^2_adj: 0.020

Log-likelihood: -383.501 	 AIC: 779.003	 BIC: 800.681

Fixed effects:

            Estimate  2.5_ci  97.5_ci     SE   DF  T-stat  P-val Sig
Intercept     -0.000  -0.118    0.118  0.060  268  -0.000  1.000    
PSE_subt       0.049  -0.116    0.214  0.084  268   0.583  0.560    
range_subt     0.043  -0.117    0.203  0.081  268   0.529  0.598    
bias_subt      0.131  -0.028    0.291  0.081  268   1.621  0.106    
range_pf      -0.214  -0.352   -0.076  0.070  268  -3.043  0.003  **
bias_pf       -0.043  -0.161    0.076  0.060  268  -0.706  0.481    
